# 01_forward_pass

This notebook walks through a GPT forward pass step by step. Use each section to add and run the code for that part.

## 1. Introduction

# GPT From Scratch — Complete Forward Pass

This notebook traces one sentence through the entire GPT model.

Goal:
- Understand every tensor.
- Verify every tensor shape.
- Verify implementation correctness.
- Build intuition before training.

## 2. Imports

Import the libraries you need for the forward pass.

In [ ]:
from warnings import filterwarnings
filterwarnings("ignore")

In [ ]:
import sys
print(sys.executable)

In [ ]:
# Optional: install plotting libraries if needed (run outside notebook if preferred)
# !pip install matplotlib

In [ ]:
import torch

print(torch.__version__)

In [ ]:
# ============================================================
# Imports
# ============================================================

from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

# Optional plotting; import if available
try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

from tokeniser.bpe_tokeniser import BPETokenizer
from model.gpt import GPT

## 3. Configuration

Define model and data configuration values.

# Model Configuration

Before tracing the forward pass, we define the architecture of the GPT model.

These hyperparameters determine:

- vocabulary size
- embedding dimension
- number of attention heads
- number of transformer blocks
- maximum context length
- positional encoding strategy

All subsequent tensor shapes in this notebook depend on these values.

In [ ]:
config = {
    "vocab_size": 4096,
    "n_embd": 384,
    "n_head": 6,
    "n_layer": 6,
    "block_size": 128,
    "dropout": 0.1,
    "pos_encoding": "rope",
}

In [ ]:
print("=" * 50)
print("GPT Configuration")
print("=" * 50)

for k, v in config.items():
    print(f"{k:<15}: {v}")
head_dim = config["n_embd"] // config["n_head"]

print(f"\nHead Dimension : {head_dim}")

## 4. Load Tokenizer

Load or initialize the tokenizer for tokenization.

# Step 1 — Load the Tokenizer

A GPT model cannot understand raw text directly.

The tokenizer converts text into a sequence of integer token IDs using the
Byte Pair Encoding (BPE) vocabulary that we trained on TinyStories.

**Input**

- Raw text

**Output**

- Integer token IDs

In this project, the tokenizer vocabulary and merge rules are loaded from:

- `vocab.json`
- `merges.json`

In [ ]:
# ============================================================
# Load Tokenizer
# ============================================================

VOCAB_PATH = Path("vocab.json")
MERGES_PATH = Path("merges.json")

tokenizer = BPETokenizer()
tokenizer.load(VOCAB_PATH, MERGES_PATH)

print("Tokenizer loaded successfully.")
print(f"Vocabulary Size : {len(tokenizer.token_to_id)}")
print(f"Number of BPE Merges : {len(tokenizer.merges)}")
print(f"Special Tokens : {tokenizer.special_tokens}")

## Observation

The tokenizer is now ready to convert text into token IDs.

Notice that:

- The vocabulary maps tokens to integer IDs.
- The merge rules determine how characters are combined into subword tokens.
- The GPT model never sees raw text—it only processes these integer token IDs.

## 5. Tokenize Input

Convert a sample text input into token IDs.

# Step 2 — Tokenize the Input Text

The GPT model cannot process raw text directly.

The tokenizer converts the input sentence into a sequence of integer token IDs
using the Byte Pair Encoding (BPE) vocabulary learned during training.

In this step, we will verify that:

- The input sentence is tokenized correctly.
- The tokenizer produces integer IDs.
- Decoding those IDs reconstructs the original sentence.

## Observation

### What?

The tokenizer converted the input sentence into **9 subword tokens**, and each
subword token was mapped to a unique integer ID.

### Why?

Neural networks cannot process raw text directly. They require numerical inputs,
so every token must be represented by an integer before it can be processed by
the embedding layer.

### How?

The tokenizer starts by splitting each word into characters and then repeatedly
applies the learned BPE merge rules stored in `merges.json`. Once no more merges
can be applied, each resulting subword token is looked up in `vocab.json` and
replaced with its corresponding integer ID.

At this stage, the model has **no understanding of language**. It only receives
a sequence of integer indices.

In [ ]:
# ============================================================
# Tokenize Input Sentence
# ============================================================

text = "he is a cute little boy"

token_ids = tokenizer.encode(text)
decoded_text = tokenizer.decode(token_ids)

print("=" * 60)
print("Input Text")
print("=" * 60)
print(text)

print("\n" + "=" * 60)
print("Token IDs")
print("=" * 60)
print(token_ids)

print("\n" + "=" * 60)
print("Decoded Text")
print("=" * 60)
print(decoded_text)

print("\n" + "=" * 60)
print("Statistics")
print("=" * 60)
print(f"Number of Tokens : {len(token_ids)}")

In [ ]:
print("=" * 60)
print("Token Mapping")
print("=" * 60)

for token_id in token_ids:
    token = tokenizer.id_to_token[token_id]
    print(f"{token:<20} -> {token_id}")

## Observation

The tokenizer converted the input sentence into **9 subword tokens**.

Each subword token is mapped to a unique integer ID.

These integer IDs are the actual inputs to the GPT model.

At this stage:

- The model has no understanding of meaning.
- It only receives integer indices.
- The next step is to convert these indices into dense embedding vectors.

## 6. Token Embeddings

Map token IDs to embeddings.

In [ ]:
model = GPT(config)
model.token_embedding
model.token_embedding.weight.shape

In [ ]:
token_tensor = torch.tensor(token_ids, dtype=torch.long)
token_tensor = token_tensor.unsqueeze(0)
print(token_tensor.shape)

In [ ]:
# ============================================================
# Convert Token IDs to a Tensor
# ============================================================

token_tensor = torch.tensor(token_ids, dtype=torch.long).unsqueeze(0)

print(f"Token Tensor Shape : {token_tensor.shape}")
print(token_tensor)

# ============================================================
# Embedding Lookup
# ============================================================

embedded_tokens = model.token_embedding(token_tensor)

print(f"\nEmbedding Shape : {embedded_tokens.shape}")

print("\nSemantic Meaning")
print(f"Batch Size          : {embedded_tokens.shape[0]}")
print(f"Sequence Length     : {embedded_tokens.shape[1]}")
print(f"Embedding Dimension : {embedded_tokens.shape[2]}")

In [ ]:
embedded_tokens[0, 1]

In [ ]:
embedded_tokens.shape

In [ ]:
print("Mean :", embedded_tokens.mean().item())
print("Std  :", embedded_tokens.std().item())

### Although nn.Embedding behaves like a lookup table during the forward pass, its rows are trainable parameters that are updated through backpropagation just like the weights of a neural network.

## 7. Positional Encoding

Add position information to token embeddings.

In [ ]:
from model.positional_encoding import (
    SinusoidalEncoding,
    LearnedPositionalEncoding,
    RotaryPositionalEmbedding,
)

In [ ]:
# ============================================================
# Positional Encoding — RoPE (Rotary Positional Embedding)
# ============================================================

# We use RoPE in this notebook. RoPE is applied inside the attention
# mechanism, so we DO NOT add positional embeddings to the token
# embeddings here. Keep the token embeddings unchanged; rotary
# transforms will be applied to queries/keys inside attention.

rotary_emb = RotaryPositionalEmbedding(
    head_dim=config["n_embd"] // config["n_head"],
    max_seq_len=config["block_size"]
)

print("Rotary positional embedding instantiated:", rotary_emb)

# For the forward-pass trace we keep the variable name the rest of the
# notebook expects but do not change the embedding values here.
embedded_with_position = embedded_tokens
print("embedded_with_position shape:", embedded_with_position.shape)

## 8. Transformer Block (Deep Dive)

Implement and inspect each step of one transformer block.

### ├── LayerNorm
### ├── Q K V
### ├── RoPE
### ├── Attention Scores
### ├── Mask
### ├── Softmax
### ├── Weighted Sum
### ├── Output Projection
### ├── Residual
### ├── LayerNorm
### ├── FFN
### └── Residual


In [ ]:
# ============================================================
# Step 5.1 — First LayerNorm
# ============================================================

first_block = model.blocks[0]

normalized_embeddings = first_block.ln1(embedded_tokens)

print("Input Shape:")
print(embedded_with_position.shape)

print("\nAfter LayerNorm:")
print(normalized_embeddings.shape)

print("\nShapes identical?")
print(embedded_with_position.shape == normalized_embeddings.shape)

In [ ]:
print("Mean :", normalized_embeddings.mean().item())
print("Std  :", normalized_embeddings.std().item())

In [ ]:
# ============================================================
# Verify LayerNorm
# ============================================================

print("Before LayerNorm")
print(f"Mean : {embedded_with_position.mean().item():.6f}")
print(f"Std  : {embedded_with_position.std().item():.6f}")

print("\nAfter LayerNorm")
print(f"Mean : {normalized_embeddings.mean().item():.6f}")
print(f"Std  : {normalized_embeddings.std().item():.6f}")

In [ ]:
# ============================================================
# Step 5.2 — Compute Q, K and V
# ============================================================

first_attn = first_block.attn

qkv = first_attn.c_attn(normalized_embeddings)

print("Input Shape:")
print(normalized_embeddings.shape)

print("\nCombined QKV Shape:")
print(qkv.shape)

q, k, v = qkv.split(config["n_embd"], dim=2)

print("\nQuery Shape :", q.shape)
print("Key Shape   :", k.shape)
print("Value Shape :", v.shape)

In [ ]:
# ============================================================
# Step 5.3 — Split into Multiple Attention Heads
# ============================================================

B, T, C = q.shape

q = q.view(B, T, first_attn.n_head, first_attn.head_dim).transpose(1, 2)
k = k.view(B, T, first_attn.n_head, first_attn.head_dim).transpose(1, 2)
v = v.view(B, T, first_attn.n_head, first_attn.head_dim).transpose(1, 2)

print("Query Shape :", q.shape)
print("Key Shape   :", k.shape)
print("Value Shape :", v.shape)

In [ ]:
print("Batch Size        :", q.shape[0])
print("Number of Heads   :", q.shape[1])
print("Sequence Length   :", q.shape[2])
print("Head Dimension    :", q.shape[3])

In [ ]:
# ============================================================
# Step 5.4 — Apply Rotary Positional Embedding
# ============================================================

q, k = first_attn.rotary_emb(q, k)

print("Query Shape :", q.shape)
print("Key Shape   :", k.shape)

In [ ]:
import math

# ============================================================
# Step 5.4 — Compute Raw Attention Scores
# ============================================================

att = (q @ k.transpose(-2, -1)) * (
    1.0 / math.sqrt(first_attn.head_dim)
)

print("Attention Score Shape:")
print(att.shape)

In [ ]:
# ============================================================
# Step 5.5 — Apply Causal Mask
# ============================================================

masked_att = att.masked_fill(
    first_attn.bias[:, :, :T, :T] == 0,
    float("-inf")
)

print("Masked Attention Shape:")
print(masked_att.shape)

In [ ]:
# ============================================================
# Step 5.6 — Softmax
# ============================================================

attention_weights = torch.softmax(masked_att, dim=-1)

print("Attention Weights Shape:")
print(attention_weights.shape)

print("\nSum across last dimension:")
print(attention_weights.sum(dim=-1))

In [ ]:
# ============================================================
# Step 5.7 — Attention Dropout
# ============================================================

attention_weights = first_attn.attn_dropout(attention_weights)

In [ ]:
# ============================================================
# Step 5.8 — Weighted Sum
# ============================================================

y = attention_weights @ v

print("Context Shape:")
print(y.shape)

In [ ]:
# ============================================================
# Step 5.9 — Merge Attention Heads
# ============================================================

y = (
    y.transpose(1, 2)
     .contiguous()
     .view(B, T, C)
)

print("Merged Shape:")
print(y.shape)

In [ ]:
# ============================================================
# Step 5.10 — Output Projection (manual) and Deterministic Comparison
# Recompute attention manually under eval() to ensure dropout is disabled
# and RoPE is applied exactly as in the module.
# ============================================================

# Put model (and submodules) into evaluation mode to disable dropout
model.eval()

# Recompute QKV from the normalized embeddings
qkv = first_attn.c_attn(normalized_embeddings)
q, k, v = qkv.split(config["n_embd"], dim=2)

# Reshape into heads
B, T, C = q.shape
q = q.view(B, T, first_attn.n_head, first_attn.head_dim).transpose(1, 2)
k = k.view(B, T, first_attn.n_head, first_attn.head_dim).transpose(1, 2)
v = v.view(B, T, first_attn.n_head, first_attn.head_dim).transpose(1, 2)

# Apply RoPE if present
if getattr(first_attn, "rotary_emb", None) is not None:
    q, k = first_attn.rotary_emb(q, k)

# Attention scores (scaled)
att_scores = (q @ k.transpose(-2, -1)) * (1.0 / (first_attn.head_dim ** 0.5))

# Causal mask from the module (slice to sequence length)
mask = first_attn.bias[:, :, :T, :T]
att_scores = att_scores.masked_fill(mask == 0, float("-inf"))

# Softmax
att_probs = F.softmax(att_scores, dim=-1)

# Dropout uses module.eval() so it will be disabled now
att_probs = first_attn.attn_dropout(att_probs)

# Weighted sum
y_manual = att_probs @ v

# Merge heads
y_manual = y_manual.transpose(1, 2).contiguous().view(B, T, C)

# Output projection + residual dropout (disabled in eval)
y_manual = first_attn.resid_dropout(first_attn.c_proj(y_manual))

# Add the residual connection from the block input
# The attention output is added back to the original input tokens.
y_manual = embedded_tokens + y_manual

# Compute the module's forward output under eval() as well
module_output, module_att = first_attn(normalized_embeddings)

print("Max absolute difference:", (y_manual - module_output).abs().max().item())
print("Allclose:", torch.allclose(y_manual, module_output, rtol=1e-5, atol=1e-6))

# For downstream cells keep variable name `y` consistent
y = y_manual

## 8.11 Feed-Forward Network (FFN)

The transformer block uses a two-layer MLP after attention:

- LayerNorm on the attention output
- Linear expansion to `4 * n_embd`
- GELU activation
- Linear projection back to `n_embd`
- Dropout is applied in training, but it is disabled in evaluation mode
- Finally, the result is added back to the residual connection

In [ ]:
# ============================================================
# Step 5.11 — Feed-Forward Network (FFN)
# ============================================================

# Apply the second LayerNorm in the block before the FFN
ffn_input = first_block.ln2(y)
print("FFN input shape:", ffn_input.shape)

# Run the block's FFN module
ffn_output = first_block.ffn(ffn_input)
print("FFN output shape:", ffn_output.shape)

# Add the residual connection
y_after_ffn = y + ffn_output
print("Shape after FFN residual:", y_after_ffn.shape)

# Compare against the block's full output after attention + FFN
# This verifies the result after the FFN step.
block_output, block_att = first_block(embedded_tokens)
print("Max absolute difference vs full block:", (y_after_ffn - block_output).abs().max().item())
print("Allclose vs full block:", torch.allclose(y_after_ffn, block_output, rtol=1e-5, atol=1e-6))

# Use this output for downstream notebook cells
x = y_after_ffn

In [ ]:
model.train()

torch.manual_seed(42)
out1, _ = first_block(embedded_tokens)

torch.manual_seed(42)
out2, _ = first_block(embedded_tokens)

print(torch.allclose(out1, out2))

In [ ]:
# Pass through all Transformer blocks
x = embedded_tokens

for block in model.blocks:
    x, _ = block(x)

print("Output after all Transformer Blocks:")
print(x.shape)

# Final LayerNorm
final_output = model.ln_f(x)

print("\nOutput after Final LayerNorm:")
print(final_output.shape)

print("\nMean :", final_output.mean().item())
print("Std  :", final_output.std().item())

In [ ]:
# ============================================================
# Step 7 — Language Model Head (LM Head)
# ============================================================

# Project the hidden representations to vocabulary logits
logits = model.lm_head(final_output)

print("Input to LM Head Shape:")
print(final_output.shape)

print("\nLogits Shape:")
print(logits.shape)

print("\nVocabulary Size:", logits.shape[-1])

# Display logits for the first token (first 10 values)
print("\nFirst 10 logits for Token 1:")
print(logits[0, 0, :10])

In [ ]:
# ============================================================
# Step 8 — Next Token Prediction (Create Input & Target)
# ============================================================

# Original token IDs
print("Original Token IDs:")
print(token_tensor)

# Input: all tokens except the last
input_tokens = token_tensor[:, :-1]

# Target: all tokens except the first
targets = token_tensor[:, 1:]

print("\nInput Token IDs:")
print(input_tokens)

print("\nTarget Token IDs:")
print(targets)

print("\nInput Shape :", input_tokens.shape)
print("Target Shape:", targets.shape)

In [ ]:
input_tokens = token_tensor[:, :-1]
targets = token_tensor[:, 1:]

In [ ]:
# Forward pass using shifted input
logits, model_loss, _ = model(input_tokens, targets)

print("Logits Shape :", logits.shape)
print("Targets Shape:", targets.shape)

print("\nModel Loss:")
print(model_loss.item())

In [ ]:
manual_loss = F.cross_entropy(
    logits.view(-1, logits.size(-1)),
    targets.view(-1),
    ignore_index=0
)

print("Manual Loss:", manual_loss.item())
print("Model Loss :", model_loss.item())
print("Match:", torch.allclose(manual_loss, model_loss))